# Retrieval Optimization for Bedrock Managed Knowledge Bases

Bedrock Managed Knowledge Bases use **hybrid search** (keyword + semantic) by default. Unlike Customer-managed KBs, you cannot switch between HYBRID and SEMANTIC modes — hybrid is always on and optimized by the service.

This notebook demonstrates the retrieval configurations you **can** control:

### What you can configure (`managedSearchConfiguration`)

| Parameter | Description | Default |
|-----------|-------------|--------|
| `numberOfResults` | Number of chunks returned | 5 |
| `filter` | Metadata filters (equals, in, greaterThan, etc.) | None |
| `rerankingModelType` | `MANAGED` (free), `CUSTOM`, or `NONE` | `MANAGED` |
| `rerankingConfiguration` | Custom reranker model config | — |

### What Managed KBs handle automatically

- Hybrid search (keyword + semantic) — always on, not configurable
- Vector store optimization and scaling
- Managed reranking (free, on by default)

### What this notebook covers

1. **numberOfResults** — impact of retrieving more/fewer chunks
2. **Managed reranking vs no reranking** — quality comparison
3. **Keyword-heavy vs semantic queries** — how hybrid search handles both
4. **AgenticRetrieveStream** — agentic multi-hop retrieval

### Prerequisites

- AWS credentials with Bedrock and IAM permissions
- Python 3.10+

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

In [ ]:
import boto3
import sys
import time
import os
import json
import pprint

sys.path.insert(0, "../..")

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration ─────────────────────────────────────────────────────
bucket_name = f'bedrock-bmkb-retrieval-{suffix}-{account_id}'

# Embedding model — use managed default (no extra cost)
embedding_model = None

# Generation model for AgenticRetrieveStream
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'Bucket:     {bucket_name}')
print(f'Embedding:  {embedding_model or "managed default (no extra cost)"}')
print(f'Generation: {generation_model_arn}')

## Step 2 — Create KB and ingest data

In [ ]:
# Create bucket and upload
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})

# Upload Octank Financial 10K
file_to_upload = '../../synthetic_dataset/octank_financial_10K.pdf'
print(f'Uploading {file_to_upload}')
s3_client.upload_file(file_to_upload, bucket_name, 'octank_financial_10K.pdf')
print('Done.')

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=f'bmkb-retrieval-opt-{suffix}',
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')

# Ingest
time.sleep(15)
kb.start_ingestion_job()

## Step 3 — numberOfResults: Impact on retrieval quality

Compare how many results you get back and how relevance scores change.

In [ ]:
query = "What are Octank Financial's key risk factors and how do they manage them?"

for n in [3, 5, 10, 20]:
    response = kb.retrieve(query, num_results=n)
    chunks = response.get('retrievalResults', [])
    scores = [c.get('score', 0) for c in chunks]
    avg_score = sum(scores) / len(scores) if scores else 0
    min_score = min(scores) if scores else 0
    print(f'numberOfResults={n:2d} → returned={len(chunks):2d} | avg_score={avg_score:.4f} | min_score={min_score:.4f}')

print('\nTakeaway: More results means lower average relevance. Choose based on your downstream use case.')
print('  - RAG generation: 5-10 chunks (quality over quantity)')
print('  - Comprehensive search: 15-20 chunks (coverage over precision)')

## Step 4 — Managed reranking vs no reranking

Managed KBs include a free managed reranker by default (`rerankingModelType='MANAGED'`). 
Let's compare results with and without reranking using `AgenticRetrieveStream`.

In [ ]:
query = "What is Octank's revenue growth strategy and how does it compare to industry benchmarks?"

# With managed reranking (default — free)
print('=== With Managed Reranking (default, no extra cost) ===')
result_reranked = kb.agentic_retrieve_stream(
    query=query,
    model_arn=generation_model_arn,
    generate_response=True,
    reranking_model_type='MANAGED',
)
if result_reranked.get('generated_response'):
    print(f"Answer: {result_reranked['generated_response']['answer'][:300]}...")
    print(f"Chunks: {len(result_reranked['results'])} | Citations: {len(result_reranked['generated_response'].get('citations', []))}")

print()

# Without reranking
print('=== Without Reranking (NONE) ===')
result_no_rerank = kb.agentic_retrieve_stream(
    query=query,
    model_arn=generation_model_arn,
    generate_response=True,
    reranking_model_type='NONE',
)
if result_no_rerank.get('generated_response'):
    print(f"Answer: {result_no_rerank['generated_response']['answer'][:300]}...")
    print(f"Chunks: {len(result_no_rerank['results'])} | Citations: {len(result_no_rerank['generated_response'].get('citations', []))}")

## Step 5 — Hybrid search behavior: keyword vs semantic queries

Hybrid search combines keyword matching (exact terms) with semantic similarity (meaning). Let's see how it handles different query styles.

In [ ]:
# Different query types that test hybrid search behavior
queries = {
    'Exact term (keyword-heavy)': 'Octank Financial 10K filing 2024',
    'Conceptual (semantic-heavy)': 'How does the company protect against economic downturns?',
    'Mixed (keyword + semantic)': 'Octank Financial cybersecurity risk management strategy',
    'Vague/exploratory': 'Tell me about their future plans',
}

for label, query in queries.items():
    response = kb.retrieve(query, num_results=5)
    chunks = response.get('retrievalResults', [])
    scores = [c.get('score', 0) for c in chunks]
    avg_score = sum(scores) / len(scores) if scores else 0
    top_text = chunks[0]['content']['text'][:120] if chunks else 'No results'
    
    print(f'\n[{label}]')
    print(f'  Query: "{query}"')
    print(f'  Results: {len(chunks)} | Avg score: {avg_score:.4f}')
    print(f'  Top result: {top_text}...')

## Step 6 — AgenticRetrieveStream: Multi-hop retrieval

Complex queries that require decomposition into sub-queries — this is where `AgenticRetrieveStream` shines. It iteratively retrieves and reasons to build a comprehensive answer.

In [ ]:
complex_query = "Compare Octank Financial's approach to operational risk vs market risk. Which one do they consider more significant and what mitigation strategies do they use for each?"

print(f'Query: {complex_query}\n')

result = kb.agentic_retrieve_stream(
    query=complex_query,
    model_arn=generation_model_arn,
    generate_response=True,
    max_results=10,
    max_iterations=3,
)

# Show trace (sub-query decomposition)
print(f'Trace events: {len(result["traces"])}')
for t in result['traces']:
    attrs = t.get('attributes', {})
    if attrs.get('query'):
        print(f'  Sub-query: "{attrs["query"][:80]}"')

print(f'\nTotal chunks retrieved: {len(result["results"])}')

if result.get('generated_response'):
    print(f'\nGenerated Answer:\n{result["generated_response"]["answer"]}')
    print(f'\nCitations: {len(result["generated_response"].get("citations", []))}')

## Step 7 — Direct Retrieve API with managedSearchConfiguration

For full control, call the Retrieve API directly with `managedSearchConfiguration`.

In [ ]:
runtime_client = kb.get_runtime_client()

# Full managedSearchConfiguration example
response = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': 'What are the key financial metrics?'},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 10,
            'rerankingModelType': 'MANAGED',  # MANAGED (free) | CUSTOM | NONE
        }
    }
)

print(f'Results: {len(response["retrievalResults"])}')
for i, r in enumerate(response['retrievalResults'][:5], 1):
    print(f'  {i}. score={r.get("score", 0):.4f} | {r["content"]["text"][:100]}...')

## Step 8 — Cleanup

In [ ]:
# Uncomment to delete all resources
# kb.delete_kb(delete_iam=True, delete_s3_bucket=True)

## Summary

### Managed KB retrieval configuration

| What you control | How | Default |
|-----------------|-----|--------|
| Number of results | `numberOfResults` in `managedSearchConfiguration` | 5 |
| Reranking | `rerankingModelType`: MANAGED / CUSTOM / NONE | MANAGED (free) |
| Metadata filtering | `filter` in `managedSearchConfiguration` | None |
| Agentic retrieval | `AgenticRetrieveStream` with `max_iterations` | 3 iterations |

### What Managed KBs do automatically

- **Hybrid search** (keyword + semantic) — always on, not configurable
- **Vector store scaling** — managed infrastructure
- **Managed reranking** — free semantic reranker, on by default

### Managed vs Customer-managed KB search

| Feature | Managed KB | Customer-managed KB |
|---------|-----------|--------------------|
| Config key | `managedSearchConfiguration` | `vectorSearchConfiguration` |
| Search type toggle | Not available (always hybrid) | `overrideSearchType`: HYBRID / SEMANTIC |
| Reranking | MANAGED (free) / CUSTOM / NONE | CUSTOM only |
| Metadata filters | ✅ Same syntax | ✅ Same syntax |
| Implicit filters | Not available | ✅ Available |

### Key takeaways

- Managed KBs optimize retrieval automatically — no search type tuning needed
- Use `numberOfResults` to balance precision vs coverage
- Managed reranking is free and improves quality — leave it on unless benchmarking
- For complex queries, use `AgenticRetrieveStream` with higher `max_iterations`

### Documentation

- [Query configurations](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-test-config.html)
- [Retrieve API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_Retrieve.html)
- [AgenticRetrieveStream API](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agent-runtime/client/agentic_retrieve_stream.html)